In [0]:
import __init__
from src.config.config_store import *

In [0]:
pip install -U polygon-api-client

In [0]:
from polygon import RESTClient

In [0]:
helper = UCSetup(spark, dbutils)

In [0]:
landing_zone = helper.get_paths()["landing_zone_path"]
checkpoint = helper.get_paths()['checkpoint_path']
landing_zone, checkpoint

In [0]:
csv_path = f"{landing_zone}/AAPL_minute_*.csv"
bronze_table = "dev.bronze.aapl_minutes"

In [0]:
import pyspark.sql.functions as sf

schema = """
    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE,
    volume LONG,
    vwap DOUBLE,
    timestamp LONG,
    transactions LONG,
    otc STRING,
    TimestampIst STRING
"""

df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("maxFilesPerTrigger", 1)
        .schema(schema)
        .load(csv_path)
        .withColumn("load_time", sf.current_timestamp())
        .withColumn("source_file", sf.col("_metadata.file_path"))
)

bronze_writer = (
    df_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{checkpoint}/aapl_minutes_bronze")
        .queryName("aapl_minutes_bronze")
        .trigger(availableNow=True)
        .toTable(bronze_table)
)

In [0]:
%sql
select *
from dev.bronze.aapl_minutes